# Milestone 4 🚩

In [2]:
import torch
import numpy as np
import random
import torchaudio
import os
import glob
from pathlib import Path

# --- SET YOUR KAGGLE PATHS ---
INPUT_BASE = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
WORKING_BASE = '/kaggle/working'

STEMS_PATH = os.path.join(INPUT_BASE, 'genres_stems')
NOISE_PATH = os.path.join(INPUT_BASE, 'ESC-50-master/audio')
OUTPUT_PATH = os.path.join(WORKING_BASE, 'synthetic_mashups/train')


def seed_everything(seed=42):
    """Locks all random seeds for absolute reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    # If using GPU
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        # Forces deterministic algorithms
        torch.backends.cudnn.deterministic = True 
        torch.backends.cudnn.benchmark = False

# Execute immediately at the top of the script
seed_everything(42)

def generate_synthetic_dataset(stems_dir, noise_dir, output_dir, samples_per_genre=50, target_sr=22050, duration=30):
    """Generates deterministic noisy mashups and saves them to /kaggle/working/."""
    genres = ["blues", "classical", "country", "disco", "hiphop",
"jazz", "metal", "pop", "reggae", "rock"
]
    target_length = target_sr * duration
    
    # Get noise files from read-only input
    noise_files = glob.glob(os.path.join(noise_dir, '**', '*.wav'), recursive=True)
    
    for genre in genres:
        # Create output directories in the writable /kaggle/working/ directory
        genre_out_dir = Path(output_dir) / genre
        genre_out_dir.mkdir(parents=True, exist_ok=True)
        
        song_folders = glob.glob(os.path.join(stems_dir, genre, '*'))
        if not song_folders: 
            print(f"Warning: No songs found for genre {genre}")
            continue
        
        for i in range(samples_per_genre):
            chosen_songs = random.sample(song_folders, 4)
            stems = []
            stem_types = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
            
            for song, stem_type in zip(chosen_songs, stem_types):
                stem_path = os.path.join(song, stem_type)
                if os.path.exists(stem_path):
                    waveform, sr = torchaudio.load(stem_path)
                    
                    # Basic Resampling check (if needed)
                    if sr != target_sr:
                        resampler = torchaudio.transforms.Resample(sr, target_sr)
                        waveform = resampler(waveform)

                    if waveform.shape[1] > target_length:
                        waveform = waveform[:, :target_length]
                    elif waveform.shape[1] < target_length:
                        waveform = torch.nn.functional.pad(waveform, (0, target_length - waveform.shape[1]))
                    stems.append(waveform)
            
            if len(stems) == 4:
                mashup = torch.stack(stems).sum(dim=0)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                
                noise_file = random.choice(noise_files)
                noise, _ = torchaudio.load(noise_file)
                
                if noise.shape[1] > target_length:
                    noise = noise[:, :target_length]
                    
                start_idx = random.randint(0, target_length - noise.shape[1])
                intensity = random.uniform(0.1, 0.4)
                
                mashup[:, start_idx:start_idx + noise.shape[1]] += (noise * intensity)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                
                # Save to /kaggle/working/
                out_path = genre_out_dir / f"mashup_{i:03d}.wav"
                torchaudio.save(str(out_path), mashup, target_sr)

# Run the generation
generate_synthetic_dataset(STEMS_PATH, NOISE_PATH, OUTPUT_PATH, samples_per_genre=50)

/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be r

**Q1)** You ran the generate_synthetic_dataset script with samples_per_genre=50. If the script executed successfully across all 10 target genres, exactly how many .wav files should now exist in your /kaggle/working/synthetic_mashups/train/ directory tree?  

**50*10 = 500**

**Q2)** Load any of your newly generated .wav files using torchaudio.load(). Given our configuration (target_sr=22050, duration=30), what is the exact tensor shape of the resulting waveform?  
Provide in tuple format, example (x,y)

In [4]:
waveform, sr = torchaudio.load("/kaggle/working/synthetic_mashups/train/blues/mashup_003.wav")
waveform.shape

torch.Size([2, 661500])

In [8]:
def extract_and_save_features(input_dir, output_dir, target_sr=22050):
    """Converts audio to Mel-spectrograms in dB and saves as PyTorch tensors."""
    mel_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=target_sr, n_fft=2048, hop_length=512, n_mels=128
    )
    amplitude_to_db = torchaudio.transforms.AmplitudeToDB()

    # Find all .wav files in the input directory
    wav_files = glob.glob(os.path.join(input_dir, '**', '*.wav'), recursive=True)
    
    if not wav_files:
        print(f"Warning: No .wav files found in {input_dir}")
        return

    for wav_path in wav_files:
        # Replicate directory structure
        rel_path = os.path.relpath(wav_path, input_dir)
        out_path = Path(output_dir) / rel_path
        out_path = out_path.with_suffix('.pt')
        
        # Ensure the target directory exists in /kaggle/working/
        out_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Process and save
        waveform, sr = torchaudio.load(wav_path)
        mel_spec = mel_transform(waveform)
        mel_spec_db = amplitude_to_db(mel_spec)
        
        torch.save(mel_spec_db, out_path)
    
    print(f"Successfully saved {len(wav_files)} feature files to {output_dir}")


INPUT_DIR = '/kaggle/working/synthetic_mashups/train'
OUTPUT_DIR = '/kaggle/working/features/train'

extract_and_save_features(INPUT_DIR, OUTPUT_DIR)

Successfully saved 500 feature files to /kaggle/working/features/train


**Q3)** Using the extract_and_save_features script with n_fft=2048, hop_length=512, and n_mels=128, you converted the waveforms into .pt files.   

If you load one of these pre-computed tensors using torch.load(), what is its exact dimension shape?

In [9]:
random_mashup_tensor = torch.load("/kaggle/working/features/train/blues/mashup_000.pt")
random_mashup_tensor.shape

torch.Size([2, 128, 1292])

In [14]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class PrecomputedFeatureDataset(Dataset):
    def __init__(self, features_dir):
        self.files = glob.glob(os.path.join(features_dir, '**', '*.pt'), recursive=True)
        self.genres = sorted(['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock'])
        self.genre_to_idx = {g: i for i, g in enumerate(self.genres)}

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        # Extract genre from the directory name
        genre = Path(file_path).parent.name
        label = self.genre_to_idx[genre] # converts genre name to a numerically encoded value
        
        # Load precomputed tensor
        feature = torch.load(file_path)
        return feature, label

class CRNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        
        # TODO 1: Define the CNN Backbone using nn.Sequential
        # Expect an input of shape (Batch_Size, 1, 128, Time_Steps)
        # Block 1: Conv2d(1 -> 32, kernel=3, padding=1) -> BatchNorm2d -> ReLU -> MaxPool2d(2)
        # Block 2: Conv2d(32 -> 64, kernel=3, padding=1) -> BatchNorm2d -> ReLU -> MaxPool2d(2)
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        
        # TODO 2: Define the RNN Component
        # Hint: Calculate the flattened feature size coming out of your CNN.
        # Original Mels = 128. After two MaxPool2d(2) layers, Mels = 128 / 4 = 32.
        # Channels = 64. So, Input Size = Channels * Mels = 2048.
        # Create a 1-layer Bidirectional LSTM with hidden_size=64 and batch_first=True.
        self.lstm = nn.LSTM(input_size=2048, hidden_size=64, num_layers=1, batch_first=True, bidirectional=True)
        
        # TODO 3: Define the Classifier
        # Create a Fully Connected (Linear) layer. 
        # Hint: Since the LSTM is bidirectional, the input features will be hidden_size * 2.
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        """
        Input:
            x: Tensor of shape (Batch, 1, 128, Time) representing Mel-spectrograms.
        Output:
            logits: Tensor of shape (Batch, num_classes) representing unnormalized class scores.
        """
        
        # TODO 4: Pass 'x' through the CNN backbone
        # Expected shape after CNN: (Batch, Channels=64, Mels=32, Time)
        x = self.cnn(x)
        
        # TODO 5: Bridge the gap (Shape Manipulation)
        # Permute and reshape the CNN output to be compatible with the LSTM.
        # Extract b, c, f, t from the tensor shape.
        # Permute the dimensions to bring Time forward, then reshape to flatten Channels and Mels.
        # Target shape for LSTM: (Batch, Time_Steps, Channels * Mels)
        b, c, f, t = x.shape
        x = x.permute(0, 3, 1, 2).reshape(b, t, c * f)
        
        # TODO 6: Pass the reshaped sequence through the LSTM
        # The LSTM returns output and (hidden_state, cell_state). You only need the output.
        x, _ = self.lstm(x)
        
        # TODO 7: Global Max Pooling over the time dimension
        # Collapse the sequence down to a single vector using torch.max() over the time dimension (dim=1).
        # Note: torch.max returns both values and indices. You only need the values.
        x, _ = torch.max(x, dim=1)
        
        # TODO 8: Pass the pooled vector through the fully connected layer
        logits = self.fc(x)
        
        # return logits
        return logits
        # raise NotImplementedError("You need to implement the CRNN forward pass!")

**Q4)** Inside the CRNN model's forward pass, the data moves from the CNN backbone to the LSTM. If your input batch has a size of 32, what is the exact shape of the tensor immediately after the second MaxPool2d(2) layer, right before the .permute() operation?  

**(32,64,32,323)**

**Q5)** Your model uses a Bidirectional LSTM with input_size=2048 and hidden_size=64. Write a small snippet using sum(p.numel() for p in model.lstm.parameters() if p.requires_grad) to calculate the exact number of trainable parameters in the LSTM layer alone. What is that integer value?

In [15]:
model = CRNN()

In [17]:
sum(p.numel() for p in model.lstm.parameters() if p.requires_grad)

1082368

In [18]:
import torch.optim as optim
from tqdm import tqdm

# --- Dataset & DataLoader Setup ---
train_dir = '/kaggle/working/features/train'
# val_dir = '/kaggle/working/features/val' # Assuming a val directory exists

train_dataset = PrecomputedFeatureDataset(train_dir)
# val_dataset = PrecomputedFeatureDataset(val_dir)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# --- Training Configuration ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CRNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 10

# --- Training Loop ---
for epoch in range(num_epochs):
    
    # === Training Phase ===
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]", leave=False)
    for inputs, labels in train_bar:
        # Convert stereo (Batch, 2, 128, Time) to mono (Batch, 1, 128, Time)
        inputs = inputs.mean(dim=1, keepdim=True)
        
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        # Track metrics
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()
        
        train_bar.set_postfix({'loss': f"{loss.item():.4f}"})
        
    avg_train_loss = train_loss / train_total
    train_acc = 100 * train_correct / train_total
    
    # # === Validation Phase ===
    # model.eval()
    # val_loss, val_correct, val_total = 0.0, 0, 0
    
    # val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]", leave=False)
    # with torch.no_grad():
    #     for inputs, labels in val_bar:
    #         # Convert stereo to mono
    #         inputs = inputs.mean(dim=1, keepdim=True)
            
    #         inputs, labels = inputs.to(device), labels.to(device)
    #         outputs = model(inputs)
    #         loss = criterion(outputs, labels)
            
    #         # Track metrics
    #         val_loss += loss.item() * inputs.size(0)
    #         _, predicted = torch.max(outputs, 1)
    #         val_total += labels.size(0)
    #         val_correct += (predicted == labels).sum().item()
            
    #         val_bar.set_postfix({'loss': f"{loss.item():.4f}"})
            
    # avg_val_loss = val_loss / val_total
    # val_acc = 100 * val_correct / val_total
    
    # === Print Epoch Results ===
    # print(f"Epoch [{epoch+1:02d}/{num_epochs:02d}] "
    #       f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.2f}% || "
    #       f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}%")
    print(f"Epoch [{epoch+1:02d}/{num_epochs:02d}] "
          f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.2f}%")

Epoch [01/10] Train Loss: 2.1969 | Train Acc: 23.60%


Epoch [02/10] Train Loss: 1.9243 | Train Acc: 42.80%


Epoch [03/10] Train Loss: 1.7603 | Train Acc: 46.80%


Epoch [04/10] Train Loss: 1.6487 | Train Acc: 54.80%


Epoch [05/10] Train Loss: 1.5244 | Train Acc: 60.00%


Epoch [06/10] Train Loss: 1.3933 | Train Acc: 66.40%


Epoch [07/10] Train Loss: 1.2733 | Train Acc: 71.40%


Epoch [08/10] Train Loss: 1.1912 | Train Acc: 73.20%


Epoch [09/10] Train Loss: 1.0990 | Train Acc: 77.00%


Epoch [10/10] Train Loss: 1.0063 | Train Acc: 80.80%


**Q6)** When applying a 2D CNN to a Mel-spectrogram, translation invariance along the X-axis (Time) is highly beneficial, as a guitar solo at 0:15 is the same as a guitar solo at 0:25. However, why is translation invariance along the Y-axis (Mel-Frequency Bins) conceptually problematic for music?  

Moving along the Y-axis changes the tempo of the song.  

Moving a pattern up the Y-axis fundamentally changes its pitch and instrument identity (e.g., shifting a low bassline up 80 bins turns it into a high-pitched synth).  

The Y-axis represents the stereo panning (left/right speakers).  

The CNN kernel cannot mathematically move vertically across a spectrogram.  

*ANS :* **Moving a pattern up the Y-axis fundamentally changes its pitch and instrument identity (e.g., shifting a low bassline up 80 bins turns it into a high-pitched synth).**